# Libraries

In [1]:
import numpy as np
import pandas as pd
import anndata as ad

import torch

from scipy import sparse
from torch.utils.data import Dataset


import json
from pathlib import Path

from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab
from scgpt.utils import load_pretrained

import torch
from torch import nn

/software/envs/micromamba/envs/nw-huggingface/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Datasets

### Settings

In [2]:
DATASET_CONFIG = {
    "be1": {
        "path": "/projects/shared/intronic_bam/datasets/anndata/be1.h5ad",
        "label_col": "Sample",
        "gene_col": "Symbol",
        "ensembl_col": "ID",
    },
    "sc_mixology": {
        "path": "/projects/shared/intronic_bam/datasets/anndata/sc_mix.h5ad",
        "label_col": "cell_line",
        "gene_col": None,
        "ensembl_col": None,
    },
    "cord_blood": {
        "path": "/projects/shared/intronic_bam/datasets/anndata/cord_blood.h5ad",
        "label_col": "celltype",
        "gene_col": "gene_names",
        "ensembl_col": None,
    },
}

### Class

In [3]:
class SingleCellDataset(Dataset):

    def __init__(self, dataset_name, split="all"):
        config = DATASET_CONFIG[dataset_name]

        self.adata = ad.read_h5ad(
            config["path"],
            backed="r",
        )

        obs = self.adata.obs

        # QC filtering
        qc_mask = ~obs["discard"].astype(bool).to_numpy()

        # Existing train/test partition
        split_values = (
            obs["is_train"]
            .astype(str)
            .str.upper()
            .to_numpy()
        )

        if split == "train":
            split_mask = split_values == "TRUE"
        elif split == "test":
            split_mask = split_values == "FALSE"
        elif split == "all":
            split_mask = np.ones(len(obs), dtype=bool)
        else:
            raise ValueError(f"Unknown split: {split}")

        self.indices = np.flatnonzero(qc_mask & split_mask)

        # Cell labels
        self.labels = (
            obs[config["label_col"]]
            .iloc[self.indices]
            .tolist()
        )

        # Gene symbols
        if config["gene_col"] is not None:
            self.gene_names = (
                self.adata.var[config["gene_col"]]
                .astype(str)
                .tolist()
            )
        else:
            self.gene_names = self.adata.var_names.tolist()

        # Ensembl identifiers, where already available
        if config["ensembl_col"] is not None:
            self.ensembl_ids = (
                self.adata.var[config["ensembl_col"]]
                .astype(str)
                .tolist()
            )
        else:
            self.ensembl_ids = None

        self.cell_ids = self.adata.obs_names[
            self.indices
        ].tolist()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        cell_idx = int(self.indices[index])

        # Read a single cell from the raw-count layer
        row = self.adata.layers["counts"][cell_idx, :]

        if sparse.issparse(row):
            row = row.toarray().ravel()
        else:
            row = np.asarray(row).ravel()

        return {
            "counts": torch.tensor(
                row, dtype=torch.float32
            ),
            "label": self.labels[index],
            "cell_id": self.cell_ids[index],
        }

    def close(self):
        self.adata.file.close()


### Import

In [ ]:
be1 = SingleCellDataset("be1", split="all")
sc_mixology = SingleCellDataset("sc_mixology", split="all")
cord_blood = SingleCellDataset("cord_blood", split="all")

DATASETS = [be1, sc_mixology, cord_blood]

In [ ]:
for ds in DATASETS:
    print("Number of cells:", len(ds))

    cell = ds[0]

    print("Counts shape:", cell["counts"].shape)
    print("Label:", cell["label"])
    print("Cell ID:", cell["cell_id"])

    ds.close()
    print()

# Models

## scGPT

### Libraries

In [ ]:
import json
import os
from pathlib import Path
from typing import Optional, Union

import numpy as np
import scanpy as sc
import torch
from anndata import AnnData
from torch.utils.data import DataLoader, SequentialSampler
from tqdm import tqdm

from scgpt import logger
from scgpt.data_collator import DataCollator
from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab
from scgpt.utils import load_pretrained


PathLike = Union[str, os.PathLike]

### Dataset Adapter

In [ ]:
class ScGPTDatasetAdapter(Dataset):
    def __init__(self, dataset, vocab, pad_value):
        self.dataset = dataset
        self.vocab = vocab
        self.pad_value = pad_value

        # Mapping genes -> scGPT vocabulary
        self.gene_ids = np.array([
            vocab[g] if g in vocab else -1
            for g in dataset.gene_names
        ])

        self.valid_genes = self.gene_ids >= 0

        print(
            f"Matched genes: {self.valid_genes.sum()}"
            f"/{len(self.gene_ids)}"
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        counts = item["counts"].numpy()
        counts = counts[self.valid_genes]

        gene_ids = self.gene_ids[self.valid_genes]

        nonzero_idx = np.nonzero(counts)[0]

        values = counts[nonzero_idx]
        genes = gene_ids[nonzero_idx]

        # Add CLS token
        genes = np.insert(
            genes, 0, self.vocab["<cls>"]
        )

        values = np.insert(
            values, 0, self.pad_value
        )

        return {
            "id": idx,
            "genes": torch.tensor(genes, dtype=torch.long),
            "expressions": torch.tensor(values, dtype=torch.float32),
        }

### Utils

In [ ]:
def scgpt_get_batch_cell_embeddings(
    dataset,
    cell_embedding_mode: str = "cls",
    model=None,
    vocab=None,
    max_length=1200,
    batch_size=64,
    model_configs=None,
    gene_ids=None,
    use_batch_labels=False,
) -> np.ndarray:
    """
    Get the cell embeddings from all transformer layers.
    """

    if cell_embedding_mode == "cls":
        dataset = ScGPTDatasetAdapter(
            dataset=dataset,
            vocab=vocab,
            pad_value=model_configs["pad_value"]
        )

        collator = DataCollator(
            do_padding=True,
            pad_token_id=vocab[model_configs["pad_token"]],
            pad_value=model_configs["pad_value"],
            do_mlm=False,
            do_binning=True,
            max_length=max_length,
            sampling=True,
            keep_first_n_tokens=1,
        )

        data_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=SequentialSampler(dataset),
            collate_fn=collator,
            drop_last=False,
            num_workers=0,  # MODIFIED: backed HDF5
            pin_memory=True,
        )

        device = next(model.parameters()).device
        n_layers = len(model.transformer_encoder.layers)

        cell_embeddings = np.zeros(
            (len(dataset), n_layers, model_configs["embsize"]),
            dtype=np.float32
        )

        layer_outputs = []
        hooks = []

        def get_hook():
            def hook(module, inputs, output):
                layer_outputs.append(
                    output[:, 0, :].detach().clone()
                )
            return hook

        for layer in model.transformer_encoder.layers:
            hooks.append(layer.register_forward_hook(get_hook()))

        try:
            with torch.no_grad(), torch.autocast(
                device_type=device.type,
                enabled=device.type == "cuda"
            ):
                count = 0

                for data_dict in tqdm(data_loader, desc="Embedding cells"):
                    input_gene_ids = data_dict["gene"].to(device)

                    src_key_padding_mask = input_gene_ids.eq(
                        vocab[model_configs["pad_token"]]
                    )

                    layer_outputs.clear()

                    _ = model._encode(
                        input_gene_ids,
                        data_dict["expr"].to(device),
                        src_key_padding_mask=src_key_padding_mask,
                        batch_labels=data_dict["batch_labels"].to(device)
                        if use_batch_labels
                        else None,
                    )

                    embeddings = torch.stack(layer_outputs, dim=1)
                    embeddings = embeddings.float().cpu().numpy()

                    cell_embeddings[count : count + len(embeddings)] = embeddings
                    count += len(embeddings)

                    layer_outputs.clear()

        finally:
            for hook in hooks:
                hook.remove()

        cell_embeddings = cell_embeddings / np.maximum(
            np.linalg.norm(cell_embeddings, axis=2, keepdims=True),
            1e-12
        )

    else:
        raise ValueError(f"Unknown cell embedding mode: {cell_embedding_mode}")

    return cell_embeddings


def scgpt_embed_data(
    dataset,
    model_dir: PathLike,
    max_length=1200,
    batch_size=64,
    device: Union[str, torch.device] = "cuda",
    use_fast_transformer: bool = False,
    return_new_adata: bool = True,
) -> AnnData:

    if device == "cuda":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if not torch.cuda.is_available():
            print("WARNING: CUDA is not available. Using CPU instead.")

    # LOAD MODEL
    model_dir = Path(model_dir)
    vocab_file = model_dir / "vocab.json"
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    pad_token = "<pad>"
    special_tokens = [pad_token, "<cls>", "<eoc>"]

    # vocabulary
    vocab = GeneVocab.from_file(vocab_file)
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    # MODIFIED: gene matching is handled by ScGPTDatasetAdapter
    gene_ids_in_vocab = np.array([
        vocab[gene] if gene in vocab else -1
        for gene in dataset.gene_names
    ])

    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )

    with open(model_config_file, "r") as f:
        model_configs = json.load(f)

    vocab.set_default_index(vocab["<pad>"])

    model = TransformerModel(
        ntoken=len(vocab),
        d_model=model_configs["embsize"],
        nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"],
        nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"],
        n_cls=1,
        vocab=vocab,
        dropout=model_configs["dropout"],
        pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"],
        do_mvc=True,
        do_dab=False,
        use_batch_labels=False,
        domain_spec_batchnorm=False,
        explicit_zero_prob=False,
        use_fast_transformer=use_fast_transformer,
        fast_transformer_backend="flash",
        pre_norm=False,
    )

    load_pretrained(model, torch.load(model_file, map_location=device), verbose=False)
    model.to(device)
    model.eval()

    # get cell embeddings
    cell_embeddings = scgpt_get_batch_cell_embeddings(
        dataset,
        cell_embedding_mode="cls",
        model=model,
        vocab=vocab,
        max_length=max_length,
        batch_size=batch_size,
        model_configs=model_configs,
        gene_ids=None,
        use_batch_labels=False,
    )

    if return_new_adata:
        # MODIFIED: metadata come from SingleCellDataset
        obs_df = pd.DataFrame(
            {"label": dataset.labels},
            index=dataset.cell_ids
        )

        # MODIFIED: save only embeddings, not expression matrix
        result = sc.AnnData(obs=obs_df)

        for i in range(cell_embeddings.shape[1]):
            result.obsm[f"X_scGPT_layer_{i+1}"] = cell_embeddings[:, i, :]

        return result

    # MODIFIED: return array when return_new_adata=False
    return cell_embeddings

### Embeddings

In [ ]:
for ds in DATASETS:
    scgpt_emb = scgpt_embed_data(
        dataset=ds,
        model_dir="/projects/shared/intronic_bam/scGPT_supplements/save/scGPT_human",
        batch_size=32,
        use_fast_transformer=False,
        return_new_adata=True,
    )
    # scgpt_emb.write_h5ad("scgpt_all_layers.h5ad")

## Geneformer

### Libraries

In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch
import scanpy as sc

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import BertForMaskedLM

from datasets import Dataset as HFDataset
from tqdm.auto import trange
from torch.nn.utils.rnn import pad_sequence

### Dataset Adapter

In [ ]:
class GeneformerDatasetAdapter:

    def __init__(
        self,
        dataset,
        gene_token_dict,
        gene_median_dict,
        max_length=4096,
        special_token=True,
        gene_mapping_dict=None,
    ):
        self.dataset = dataset
        self.max_length = max_length
        self.special_token = special_token

        # Original genes (expected: Ensembl IDs)
        gene_names = list(dataset.gene_names)

        # Map old Ensembl IDs to canonical IDs, if requested.
        if gene_mapping_dict is not None:
            gene_names = [
                gene_mapping_dict.get(g, g)
                for g in gene_names
            ]

        # Keep genes available in Geneformer vocabulary
        valid_idx = [
            i for i, g in enumerate(gene_names)
            if g in gene_token_dict
            and g in gene_median_dict
            and gene_median_dict[g] > 0
        ]

        valid_genes = [gene_names[i] for i in valid_idx]

        # Collapse duplicated gene IDs by summing counts.
        unique_genes, inverse = np.unique(
            valid_genes, return_inverse=True
        )

        self.valid_idx = np.asarray(valid_idx, dtype=np.int64)
        self.inverse = inverse
        self.n_genes = len(unique_genes)

        self.gene_tokens = np.array(
            [gene_token_dict[g] for g in unique_genes],
            dtype=np.int64
        )

        self.gene_medians = np.array(
            [gene_median_dict[g] for g in unique_genes],
            dtype=np.float64
        )

        self.cls_token_id = None
        self.eos_token_id = None

        if special_token:
            self.cls_token_id = gene_token_dict["<cls>"]
            self.eos_token_id = gene_token_dict["<eos>"]

        if self.n_genes == 0:
            raise ValueError(
                "No genes matched Geneformer vocabulary."
            )

        print(
            f"Matched genes: {len(valid_idx)}/{len(gene_names)}"
        )
        print(f"Unique genes: {self.n_genes}")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]
        counts = item["counts"]

        if isinstance(counts, torch.Tensor):
            counts = counts.detach().cpu().numpy()

        counts = np.asarray(
            counts, dtype=np.float64
        ).ravel()

        if len(counts) != len(self.dataset.gene_names):
            raise ValueError(
                "Counts and gene_names have different lengths."
            )

        if not np.all(np.isfinite(counts)) or np.any(counts < 0):
            raise ValueError("Counts must be finite and nonnegative.")

        # Total counts before vocabulary filtering
        total_counts = counts.sum()

        if total_counts <= 0:
            raise ValueError(f"Cell {idx}: no counts.")

        # Select genes in Geneformer vocabulary
        selected_counts = counts[self.valid_idx]

        # Sum counts for genes mapped to the same Ensembl ID
        gene_counts = np.bincount(
            self.inverse,
            weights=selected_counts,
            minlength=self.n_genes,
        )

        # Geneformer normalization
        normalized = (
            gene_counts / total_counts * 10000
        ) / self.gene_medians

        # Rank expressed genes from highest to lowest
        nonzero_idx = np.flatnonzero(normalized > 0)

        sorted_idx = nonzero_idx[
            np.argsort(-normalized[nonzero_idx], kind="stable")
        ]

        tokens = self.gene_tokens[sorted_idx].tolist()

        n_special = 2 if self.special_token else 0
        max_genes = self.max_length - n_special

        if max_genes <= 0:
            raise ValueError("max_length is too small.")

        tokens = tokens[:max_genes]

        if not tokens:
            raise ValueError(
                f"Cell {idx}: no expressed genes in vocabulary."
            )

        if self.special_token:
            tokens = [
                self.cls_token_id,
                *tokens,
                self.eos_token_id,
            ]

        return {
            "input_ids": tokens,
            "length": len(tokens),
        }

    def to_hf_dataset(self):
        input_ids = []
        lengths = []
    
        for idx in range(len(self.dataset)):
            item = self[idx]
    
            input_ids.append(item["input_ids"])
            lengths.append(item["length"])
    
        return HFDataset.from_dict({
            "input_ids": input_ids,
            "length": lengths,
        })


### Utils

In [ ]:
def pad_tensor_list(
    tensor_list,
    max_len,
    pad_token_id,
    model_input_size,
):
    tensors = [
        torch.as_tensor(t, dtype=torch.long)[:model_input_size]
        for t in tensor_list
    ]

    return pad_sequence(
        tensors,
        batch_first=True,
        padding_value=pad_token_id,
    )

def mean_nonpadding_embs(embs, lengths):
    """
    Mean pooling excluding padding tokens.

    embs: (batch_size, seq_len, hidden_size)
    lengths: number of valid tokens per cell
    """
    device = embs.device
    lengths = lengths.to(device)

    positions = torch.arange(
        embs.shape[1], device=device
    ).unsqueeze(0)

    mask = positions < lengths.unsqueeze(1)

    summed = (
        embs * mask.unsqueeze(-1)
    ).sum(dim=1)

    counts = mask.sum(dim=1, keepdim=True).clamp(min=1)

    return summed / counts

In [ ]:
# extract embeddings
def get_embs(
    model,
    filtered_input_data,
    emb_mode,
    layer_to_quant,
    pad_token_id,
    forward_batch_size,
    summary_stat,
):
    model_input_size = model.config.max_position_embeddings
    total_batch_length = len(filtered_input_data)

    if summary_stat is None:
        embs_list = {}
    elif summary_stat is not None:
        # get # of emb dims
        emb_dims = pu.get_model_emb_dims(model)
        if emb_mode == "cell":
            # initiate tdigests for # of emb dims
            embs_tdigests = [TDigest() for _ in range(emb_dims)]
        if emb_mode == "gene":
            gene_set = list(
                {
                    element
                    for sublist in filtered_input_data["input_ids"]
                    for element in sublist
                }
            )
            # initiate dict with genes as keys and tdigests for # of emb dims as values
            embs_tdigests_dict = {
                k: [TDigest() for _ in range(emb_dims)] for k in gene_set
            }

    for i in trange(0, total_batch_length, forward_batch_size):
        max_range = min(i + forward_batch_size, total_batch_length)

        minibatch = filtered_input_data.select([i for i in range(i, max_range)])
        max_len = max(minibatch["length"])
        original_lens = torch.tensor(minibatch["length"]).to("cpu")
        minibatch.set_format(type="torch")

        input_data_minibatch = minibatch["input_ids"]
        input_data_minibatch = pad_tensor_list(
            input_data_minibatch, max_len, pad_token_id, model_input_size
        )

        device = next(model.parameters()).device

        attention_mask = (
            input_data_minibatch != pad_token_id
        ).long()
        
        with torch.no_grad():
            outputs = model(
                input_ids=input_data_minibatch.to(device),
                attention_mask=attention_mask.to(device),
                output_hidden_states=True,
            )

        all_hidden_states = outputs.hidden_states[1:]
        if summary_stat is None and not embs_list:
            embs_list = {
                layer_idx: []
                for layer_idx in range(len(all_hidden_states))
            }


        for layer_idx, embs_i in enumerate(all_hidden_states):
            if emb_mode == "cell":
                if cls_present:
                    non_cls_embs = embs_i[:, 1:, :]
        
                    if eos_present:
                        mean_embs = mean_nonpadding_embs(
                            non_cls_embs, original_lens - 2
                        )
                    else:
                        mean_embs = mean_nonpadding_embs(
                            non_cls_embs, original_lens - 1
                        )
                else:
                    mean_embs = mean_nonpadding_embs(
                        embs_i, original_lens
                    )
        
                if summary_stat is None:
                    embs_list[layer_idx].append(mean_embs.cpu())
                elif summary_stat is not None:
                    # update tdigests with current batch for each emb dim
                    # note: tdigest batch update known to be slow so updating serially
                    [
                        embs_tdigests[layer_idx][j].update(
                            mean_embs[i, j].item()
                        )
                        for i in range(mean_embs.size(0))
                        for j in range(emb_dims)
                    ]

        del outputs
        del all_hidden_states
        del minibatch
        del input_data_minibatch
        del embs_i
        del mean_embs

    if summary_stat is None:
        if (emb_mode == "cell") or (emb_mode == "cls"):
    
            embs_stack = {
                layer_idx: torch.cat(batch_list, dim=0)
                for layer_idx, batch_list in embs_list.items()
            }
        elif emb_mode == "gene":
            embs_stack = {
                layer_idx: pu.pad_tensor_list(
                    batch_list,
                    overall_max_len,
                    pad_token_id,
                    model_input_size,
                    1,
                    pu.pad_3d_tensor,
                )
                for layer_idx, batch_list in embs_list.items()
            }

    # calculate summary stat embs from approximated tdigests
    elif summary_stat is not None:
        if summary_stat == "mean":
            embs_stack = {
                layer_idx: torch.tensor([
                    td.trimmed_mean(0, 100)
                    for td in tdigests
                ])
                for layer_idx, tdigests in embs_tdigests.items()
            }
        
        elif summary_stat == "median":
            embs_stack = {
                layer_idx: torch.tensor([
                    td.percentile(50)
                    for td in tdigests
                ])
                for layer_idx, tdigests in embs_tdigests.items()
            }

    if summary_stat is None and emb_mode == "cell":
        embs_stack = torch.stack(
            [embs_stack[k] for k in sorted(embs_stack)],
            dim=1
        )

        embs_stack = embs_stack.float().cpu().numpy()

    return embs_stack

### Embeddings

In [ ]:
import pickle

with open("/software/apps/huggingface/hub/models--ctheodoris--Geneformer/snapshots/1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5/geneformer/token_dictionary_gc104M.pkl", "rb") as f:
    gene_token_dict = pickle.load(f)

with open("/software/apps/huggingface/hub/models--ctheodoris--Geneformer/snapshots/1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5/geneformer/gene_median_dictionary_gc104M.pkl", "rb") as f:
    gene_median_dict = pickle.load(f)

with open("/software/apps/huggingface/hub/models--ctheodoris--Geneformer/snapshots/1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5/geneformer/ensembl_mapping_dict_gc104M.pkl", "rb") as f:
    gene_mapping_dict = pickle.load(f)


In [ ]:
geneformer_adapter = GeneformerDatasetAdapter(
    dataset=be1,
    gene_token_dict=gene_token_dict,
    gene_median_dict=gene_median_dict,
    gene_mapping_dict=gene_mapping_dict,
    max_length=4096,
    special_token=True,
)

In [ ]:
filtered_input_data = geneformer_adapter.to_hf_dataset()

In [ ]:
from transformers import BertForMaskedLM

model_dir = (
    "/software/apps/huggingface/hub/"
    "models--ctheodoris--Geneformer/"
    "snapshots/04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5/"
    "Geneformer-V2-104M"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BertForMaskedLM.from_pretrained(
    model_dir,
    local_files_only=True,
    output_hidden_states=True,
)

model = model.to(device)
_= model.eval()

In [ ]:
model.eval()

cls_present = True
eos_present = True

geneformer_embeddings = get_embs(
    model=model,
    filtered_input_data=filtered_input_data,
    emb_mode="cell",
    layer_to_quant=None,
    pad_token_id=gene_token_dict["<pad>"],
    forward_batch_size=8,
    summary_stat=None,
)

print(geneformer_embeddings.shape)
# (n_cells, 12, 768)